# KLA restoration reproduction

This notebook runs the recovered residual U-Net on a **first-party synthetic, source-disjoint** corpus. It does not use the supplied Drift-Sense Space because no competition-use licence was observed. Results are pipeline evidence only, not official KLA or hidden-test performance.

Before running: in Colab, select **Runtime > Change runtime type > T4 GPU**, then run each cell in order.

## 1. Record the actual runtime

In [ ]:
import platform, torch
print({
    'python': platform.python_version(),
    'torch': torch.__version__,
    'cuda_available': torch.cuda.is_available(),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'cuda': torch.version.cuda,
})
!nvidia-smi

## 2. Upload the current repository archive

From the local repository run `git archive --format=zip --output=kla_restoration_submission.zip HEAD`, then upload that ZIP here. This avoids relying on an unpushed private GitHub branch.

In [ ]:
from google.colab import files
uploaded = files.upload()
assert 'kla_restoration_submission.zip' in uploaded, 'Upload kla_restoration_submission.zip'
!rm -rf kla-image-restoration
!unzip -q kla_restoration_submission.zip -d kla-image-restoration
%cd kla-image-restoration
!git rev-parse --short HEAD || true
!ls

## 3. Install the declared dependencies

In [ ]:
!python -m pip install -q -r requirements.txt
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'gpu', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

## 4. Construct the disclosed first-party corpus

The source split occurs by SHA-256 **before** views. The corpus uses only Gaussian noise, multiplicative speckle and downsampling, across all six orders.

In [ ]:
!python scripts/generate_clean_sem_sources.py --out data/restoration_sources --count 96 --size 768 --seed 20260817
!python scripts/materialize_restoration_data.py --source-dir data/restoration_sources --out data/kla_restoration --seed 20260817 --views-per-source 6 --crop-size 512 --scale 2
!cat data/kla_restoration/dataset_card.json

## 5. Train the fixed configuration

The model is selected by its validation PSNR only. Do not inspect or tune against the held-out `data/kla_restoration/test` split.

In [ ]:
!python train.py --config configs/submission_final.yaml --gt-dir data/kla_restoration/train/GT --noisy-dir data/kla_restoration/train/NoisyLR --set train.batch_size=16
!ls runs/kla_restoration_submission_seed20260817

## 6. Freeze validation-best weights and evaluate held-out sources once

In [ ]:
!mkdir -p weights results/submission_final/examples
!cp runs/kla_restoration_submission_seed20260817/best.pth weights/final_model.pth
!cp runs/kla_restoration_submission_seed20260817/resolved_config.yaml weights/final_model.config.yaml
!sha256sum weights/final_model.pth | tee weights/final_model.sha256
!python evaluate.py --gt-dir data/kla_restoration/test/GT --noisy-dir data/kla_restoration/test/NoisyLR --checkpoint weights/final_model.pth --output-dir results/submission_final --save-restored results/submission_final/examples --split all --eval-mode official
!cat results/submission_final/summary.json

## 7. Exercise the evaluator-facing inference CLI

In [ ]:
!rm -rf submission_smoke
!python inference.py --input_dir data/kla_restoration/test/NoisyLR --output_dir submission_smoke --checkpoint weights/final_model.pth --scale 2 --out-ext .png --report submission_smoke/report.json
!cat submission_smoke/report.json

## 8. Archive reproducibility artifacts

Download this archive. Copy its measured device/runtime values into `results/submission_final/colab_execution.json` only after this cell finishes successfully.

In [ ]:
import json, hashlib, pathlib, datetime
artifact = pathlib.Path('kla_restoration_artifacts.zip')
!zip -qr $artifact weights results/submission_final runs/kla_restoration_submission_seed20260817 submission_smoke data/kla_restoration/dataset_card.json data/kla_restoration/train_manifest.csv data/kla_restoration/val_manifest.csv data/kla_restoration/test_manifest.csv
print({'artifact': str(artifact), 'sha256': hashlib.sha256(artifact.read_bytes()).hexdigest(), 'utc_finished': datetime.datetime.now(datetime.UTC).isoformat()})
files.download(str(artifact))